# pyaegean — one line of Homer, all the way down

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/getting-started.ipynb)

**pyaegean** is a specialist toolkit for Ancient Greek and the Aegean syllabic scripts (Linear A, Linear B, Cypriot, and Cypro-Minoan). Instead of touring its functions in a vacuum, this notebook does what a scholar actually does: sit down with real text and read it.

We take **one** famous line — the opening of Homer's *Odyssey* — and walk it down the whole linguistic ladder, one small composable function at a time:

> text → syllables → accent → **metre** → part-of-speech → **morphology** → dictionary meaning → **dependency tree**

Then we turn the same toolkit on an *undeciphered* script, Linear A, where it gives **leads for a human**, never answers.

**How this notebook is staged**
- **Act 1 & 3 (the core) run 100% offline** — no downloads, no API key. They complete in seconds and are the spine of the tutorial.
- **Act 2 (opt-in)** switches on four heavier backends that *download data or train a model*. Every one is marked ⬇️ **HEAVY (OPTIONAL)** with its cost and gated behind a single `RUN_HEAVY` switch — the notebook runs top-to-bottom and is complete **without** them.

> **All Linear A interpretation is EXPLORATORY.** The script is undeciphered; these tools surface *leads for a human expert*, never ground truth. The library labels this everywhere, and so do we.

New to Python? You don't need to install anything — press **Open in Colab** above and run each cell top to bottom with Shift+Enter.

In [ ]:
# pyaegean works fully offline once installed. This installs it the first time
# you open the notebook in Colab; locally it's a no-op if it's already present.
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__)
print("scripts :", aegean.registered_scripts())
# ['cypriot', 'cyprominoan', 'greek', 'lineara', 'linearb'] — all four Aegean scripts + Greek

# Tip: pyaegean's objects (Corpus, Document, analysis results) render as rich
# tables/cards in a notebook. Outside Jupyter they're ordinary Python values.

In [ ]:
# ── The one switch for the optional, downloading backends ────────────────
# Leave this False to keep the whole notebook offline. Flip it to True (on decent
# wifi) to run the heavy cells in Acts 2 and 4 and the appendix:
#   * use_treebank()           ~15 MB prebuilt bundle -- attested lemmas + gold POS/morphology
#   * use_neural_lemmatizer()  ~232 MB download -- seq2seq lemmas for UNSEEN forms (76.3%)
#   * use_lsj()                ~15 MB prebuilt index  -- full Liddell-Scott-Jones glossing
#   * use_parser()             prebuilt model (in the same ~15 MB bundle as the treebank)
#   * aegean.load('damos')     ~2 MB -- the full Linear B corpus (Act 4)
# The treebank/LSJ/parser artifacts are small project-hosted prebuilt assets; they are
# built locally from the 75/270 MB upstream sources only if an asset is unreachable.
# Each fetches once, then caches. Every other cell runs offline with no downloads.
#   * the appendix shelf      I.Sicily ~7 MB · the DDbDP database ~206 MB
#   * use_neural_pipeline()   ~173 MB joint model (the [neural] extra) · AI translate (API key)
RUN_HEAVY = False

print("Heavy/optional cells are",
      "ON — first run downloads." if RUN_HEAVY
      else "OFF — the offline core still runs in full.")

## Act 1 · Reading one line of Homer (offline)

The Greek pipeline is a set of **small, independent functions** you compose yourself — and the whole of this act is **offline**, no API key. We'll take Homer's opening line and read it the way a philologist does.

> **No Greek keyboard?** Type **Beta Code** (the TLG/Perseus ASCII convention) and convert. Round-trips are NFC-safe, so you can enter any line below from a normal keyboard.

In [ ]:
from aegean import greek

# Beta Code in, Unicode out (and back) — verified round-trips:
print(greek.betacode_to_unicode("a)/ndra"))   # ἄνδρα
print(greek.unicode_to_betacode("λόγος"))     # lo/gos   (context-sensitive final ς)

LINE = "ἄνδρα μοι ἔννεπε, Μοῦσα, πολύτροπον, ὃς μάλα πολλὰ"   # Odyssey 1.1
print(greek.tokenize_words(LINE))
# ['ἄνδρα', 'μοι', 'ἔννεπε', 'Μοῦσα', 'πολύτροπον', 'ὃς', 'μάλα', 'πολλὰ']

### Rungs 1–2 · Syllables and accent

Syllabification respects diphthongs, *muta cum liquida* clusters, and doubled consonants. Accent analysis classifies the word in the traditional scheme (oxytone / paroxytone / properispomenon …) and gives you structured fields, not just a string.

In [ ]:
print(greek.syllabify("ἄνδρα"))         # ['ἄν', 'δρα']
print(greek.syllabify("πολύτροπον"))    # ['πο', 'λύ', 'τρο', 'πον']

info = greek.accentuation("Μοῦσα")
print(info.accent_type, info.position_from_end, info.classification)
# circumflex 2 properispomenon  — a circumflex on a long penult

### Two more offline tools · placing the accent, expanding contractions

`accentuation` *reads* the accent already written on a word; **`place_accent`** *predicts* one from the accentuation laws (recessively for verbs, persistently for nominals), and stays honest about vowel length: a *dichronon* (α/ι/υ, whose quantity isn't fixed by spelling) is flagged, not guessed. A second zero-dependency utility, **`resolve_sandhi`**, expands the surface contractions of running text (crasis, elision, movable nu) back to their underlying words.

In [ ]:
# place_accent PREDICTS the accent (accentuation only READS one already written):
print(greek.place_accent("λυω", recessive=True).form)        # λύω    — a verb accents recessively
print(greek.place_accent("παιδευε", recessive=True).form)    # παίδευε — back to the antepenult
ap = greek.place_accent("ἀνθρωπου", recessive=False, lemma="ἄνθρωπος")
print(ap.form, "·", ap.classification)                        # ἀνθρώπου · paroxytone (persistent)

# Honest about vowel length: an undetermined dichronon (α/ι/υ) is FLAGGED, not guessed —
# pass the vowel length (or a lemma) to resolve it:
print(greek.place_accent("λυε", recessive=True).certain)                        # False — penult υ undetermined
print(greek.place_accent("λυε", recessive=True, penult_length="short").certain) # True

# resolve_sandhi expands crasis / elision / movable-nu back to the underlying words:
for tok in ("κἀγώ", "τἀμά", "ταῦτ'", "οὐκ"):
    r = greek.resolve_sandhi(tok)
    print(f"  {tok:7} → {' + '.join(r.words):11} ({r.kind})")
# κἀγώ → καί + ἐγώ (crasis) · τἀμά → τὰ + ἐμά (crasis) · ταῦτ' → ταῦτα (elision) · οὐκ → οὐ (movable-nu)

### Rung 3 · Scan the metre

The *Odyssey* is dactylic hexameter. The scanner resolves each syllable's quantity **in context** — across word boundaries, applying *correptio*, treating muta-cum-liquida as the genuine ambiguity it is — then reports the feet and the **caesura** (the line's main pause). Notation you'll know from any commentary: **—** heavy, **⏑** light, **×** *anceps* (the "either" final syllable).

In [ ]:
sc = greek.scan_hexameter(LINE)
print(sc.pattern)                  # —⏑⏑|—⏑⏑|—⏑⏑|—⏑⏑|—⏑⏑|—×   (five dactyls)
print([f.name for f in sc.feet])   # ['dactyl','dactyl','dactyl','dactyl','dactyl','final']
print("caesura:", sc.caesura, "before syllable", repr(sc.syllables[sc.caesura_index]))
# caesura: trochaic before syllable 'πο'

### An honest scanner

Synizesis (two written vowels read as one syllable, as in `Πηληϊάδεω`) is **lexical, never inferred**: the scanner applies it only where the curated lexicon attests it, so *Iliad* 1.1 scans. And when a line genuinely cannot scan, the scanner **declines rather than forcing a fit**: it raises `ScansionError`. A tool that tells you when it *can't* is one you can trust when it does. To see where a line is genuinely ambiguous *before* a metre is imposed, ask `syllable_options`.

In [ ]:
scan = greek.scan_hexameter("μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος")   # Iliad 1.1
print([f.name for f in scan.feet])
# ['dactyl', 'dactyl', 'spondee', 'dactyl', 'dactyl', 'final']

try:
    greek.scan_hexameter("ἐν ἀρχῇ ἦν ὁ λόγος")                     # prose: nothing to force
except greek.ScansionError as e:
    print("declined:", str(e)[:62], "…")
# declined: line does not scan as dactylic hexameter (7 syllables) …

# Inspect a syllable's *possible* quantities (πα before muta-cum-liquida: either):
print(greek.syllable_options("πατρός"))
# [('πα', ['heavy', 'light']), ('τρός', ['light'])]

### Optional aside · How did it *sound*?

`to_ipa` gives a reconstructed transcription for two periods — `"attic"` (Classical, default) and `"koine"` (Hellenistic). **Reconstructed and approximate**: several values (vowel quality, the date of iotacism) are scholarly judgement calls.

In [ ]:
print(greek.to_ipa("θεός"))           # tʰeos   (Attic: aspirated θ, voiceless)
print(greek.to_ipa("θεός", "koine"))  # θeos    (Koine: θ has fricativised)
print(greek.to_ipa("καί", "koine"))   # ke      (Koine iotacism: αι → /e/)

### Rung 4 · Parts of speech and morphology — meeting the baseline honestly

The **offline baseline** is rule-based and deliberately honest. Closed-class words (article, prepositions, conjunctions, particles, pronouns, the copula) are tagged reliably from a lexicon; open-class words fall back to a light suffix heuristic, and `analyze` returns *every* reading an ending licenses — Greek inflection is richly ambiguous, and that ambiguity is the linguistic reality.

Watch it tag the verb `ἔννεπε` and the adverb `μάλα` as **NOUN**, and reconstruct the *wrong, unaccented* lemma for `ἄνδρα` with `lemma_certain=False`. That's the baseline being honest, not broken — and the cliffhanger Act 2 will pay off.

In [ ]:
# Baseline POS on our line — honest about its limits:
for w, p in greek.pos_tags(LINE):
    print(f"  {w:11} {p}")
# ἔννεπε / μάλα fall back to NOUN; the closed-class ὃς is correctly PRON.

# A REGULAR form analyses cleanly — all the readings the -ον ending licenses:
print("\nλόγον readings:")
for a in greek.analyze("λόγον"):
    print("  ", a)
# λόγος [NOUN acc sg masc] / [acc sg fem] / [nom sg neut] / [acc sg neut] / [voc sg neut]

# ἄνδρα is the acc. sg. of the irregular 3rd-declension ἀνήρ — beyond the rules.
# The lemma comes back UNACCENTED and flagged uncertain: the engine's "I guessed":
print("\nἄνδρα on the baseline:")
for a in greek.analyze("ἄνδρα"):
    print("  ", a, "| certain:", a.lemma_certain)
# ανδρα [NOUN ...] | certain: False   ← unaccented + wrong; ἄνδρα is acc sg of ἀνήρ
# The lesson: read `lemma_certain`. The fix for attested forms is one switch away →

## Act 2 · The payoff: opt-in backends that turn guesses into scholarship

Everything above ran **offline in seconds**. pyaegean offers **five opt-in** backends built from gold scholarly data: three resolve *attested* forms (treebank, the dictionary registry, parser), one — the neural lemmatizer — *generates* lemmas for **unseen** forms, and one — the **neural pipeline** — does the whole job (tagging, morphology, parsing, lemmas) with one jointly-trained model, at state-of-the-art accuracy on the standard Ancient Greek benchmarks. Each is one function call, then cached, but each downloads data or trains a model on first use, so they're isolated here and gated behind `RUN_HEAVY`.

| Backend | Call | Gives you | First-run cost |
|---|---|---|---|
| **Treebank** (AGDT v2.1, CC BY-SA) | `greek.use_treebank()` | attested accented lemmas + gold POS/morphology | ~75 MB download |
| **Neural lemmatizer** (GreTa seq2seq, `[neural]`) | `greek.use_neural_lemmatizer()` | generated lemmas for *unseen* forms (76.3%) | ~232 MB download |
| **Dictionary registry** (LSJ, Middle Liddell, Cunliffe, Abbott-Smith) | `greek.use_lsj()` / `greek.use_lexicon(id)` | glosses from any dictionary, plus Logeion deep-links | ~2–15 MB per dictionary |
| **Dependency parser** (arc-eager + perceptron, trained on AGDT) | `greek.use_parser()` | dependency trees with Prague labels | ~2-3 min train, then ~4 MB |
| **Neural pipeline** (joint model, `[neural]`) | `greek.use_neural_pipeline()` | POS + morphology + **UD trees** + lemmas in one pass — the package's most accurate option (97.0 UPOS / 94.3 lemma / 90.2 UAS, UD Perseus test) | ~173 MB download |

Each is fetched to your cache, **never bundled**, and has a matching `disable_*()`. The cells below are guarded by `RUN_HEAVY`: if it's `False` they print what they *would* do and skip the download, and the rest of the notebook still works.

> ⬇️ **HEAVY (OPTIONAL) — Treebank backend.** First run fetches the **~15 MB prebuilt AGDT bundle** (built locally from the ~75 MB treebank only if the asset is unreachable); afterwards it's instant. Runs only if `RUN_HEAVY = True`.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the prebuilt treebank lexicon (~15 MB).")
else:
    greek.use_treebank()        # one-time ~15 MB prebuilt fetch, then cached

    # The same line now tags the open-class words correctly:
    print("POS  :", greek.pos_tags(LINE)[:5])
    # [('ἄνδρα','NOUN'), ('μοι','PRON'), ('ἔννεπε','VERB'), (',','PUNCT'), ('Μοῦσα','NOUN')]

    # The ἄνδρα cliffhanger, resolved — attested, correctly accented:
    print("ἄνδρα →", greek.lemmatize("ἄνδρα"))     # ἀνήρ
    for w in ("ἔφη", "γυναικός", "πόλεως"):
        print(f"  {w:9} → {greek.lemmatize(w)}")    # φημί · γυνή · πόλις

    # Put a NUMBER on the lift, on a hand-authored independent gold set:
    from aegean.greek import benchmark
    benchmark.compare_modes()   # toggles the backend internally, leaving it disabled
    # baseline : lemma  28% (5/18)   · pos  50% (10/20)
    # treebank : lemma 100% (18/18)  · pos 100% (20/20)

    greek.use_treebank()        # re-enable it for the cells below (instant once cached)

### The unseen-form lever · the neural lemmatizer

The treebank aces *attested* forms, but on a genuinely **unseen** word the offline baseline only goes so far: its rule layer recovers the regular paradigms (`νόμου → νόμος`) without a lookup, yet an irregular third-declension form like `ἄνδρα` (accusative of `ἀνήρ`) has no nominative to rebuild and comes back unchanged. pyaegean's `[neural]` backend closes that gap: a **GreTa** (Ancient-Greek T5) seq2seq, exported to int8 ONNX and run **torch-free** (numpy + onnxruntime), that *generates* the lemma. It reaches **76.3% on unseen forms** — the hardest case, where recovering a lemma means an internal stem/accent change, not just a suffix swap. It ships as a **hybrid**: the bundled gold lookup answers seen forms instantly, the seq2seq handles the rest, and `greek.lemmatize` cascades treebank → neural → edit-tree → seed.

> ⬇️ **HEAVY (OPTIONAL) — neural lemmatizer.** First run downloads the **~232 MB** ONNX model (the `[neural]` extra) and caches it; afterwards it's instant. Runs only if `RUN_HEAVY = True`.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to download the neural lemmatizer (~232 MB, the [neural] extra).")
else:
    %pip install -q "pyaegean[neural]"   # onnxruntime + tokenizers (no torch)
    greek.use_neural_lemmatizer()        # one-time ~232 MB model download, then cached

    # Unseen, inflected forms the seed table can't reach — the seq2seq *generates* the lemma:
    for w in ("ἐποίησεν", "ἔγραψεν", "ἀπέθανεν"):
        print(f"  {w:11} → {greek.lemmatize(w)}")
    # e.g. ἐποίησεν → ποιέω   (generated, not a table lookup)

    greek.disable_neural_lemmatizer()    # back to the offline cascade

### What does *λόγος* actually mean?

> ⬇️ **HEAVY (OPTIONAL) — dictionaries.** First run fetches a small prebuilt index per dictionary (the LSJ index is ~15 MB; Middle Liddell / Cunliffe / Abbott-Smith are 0.1–2.3 MB). Runs only if `RUN_HEAVY = True`.

`use_lsj()` is one backend in a **lexicon registry**. `greek.use_lexicon(id)` activates any of **Middle Liddell** (concise classical), **Cunliffe** (Homeric), **Abbott-Smith** (New Testament), or **LSJ**, and `greek.gloss(word, dictionary=id)` glosses from the one you pick. Glossing **composes** with the lemmatizer: hand it an inflected form and it tries the form, then lemmatizes (using the treebank above if you switched it on) and retries — so `ἀνδρός` resolves to *ἀνήρ* automatically. For dictionaries pyaegean does not host (Autenrieth, Slater, …), `greek.lexicon_link(word)` builds a Logeion deep-link, and that one runs **offline**.

In [ ]:
# lexicon_link is OFFLINE — a Logeion deep-link to a word's dictionary entry (lemmatized first):
print("Logeion:", greek.lexicon_link("μῆνις"))   # a (percent-encoded) link to logeion.uchicago.edu/μῆνις

# The rest fetches small dictionary indexes; RUN_HEAVY gates it (flip the switch at the top).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the dictionary indexes (0.1–15 MB each).")
else:
    greek.use_treebank()                  # so inflected forms (ἀνδρός → ἀνήρ) resolve for glossing
    greek.use_lsj()                       # one-time ~15 MB prebuilt fetch, then cached
    greek.use_lexicon("middle-liddell")   # the concise Intermediate Lexicon
    greek.use_lexicon("cunliffe")         # Cunliffe, A Lexicon of the Homeric Dialect
    print("dictionaries:", [i.id for i in greek.lexica()])

    for w in ("ἀνδρός", "βάλλω"):
        print(f"{w:7} → {greek.gloss(w)[:60]}")   # LSJ by default; ἀνδρός is lemmatised to ἀνήρ

    # The same word from different dictionaries — choose one with dictionary=:
    print("\nμῆνις, three ways (each glosses it as 'wrath' in its own register):")
    for d in ("lsj", "middle-liddell", "cunliffe"):
        print(f"  {d:14} {greek.gloss('μῆνις', dictionary=d)[:55]}")

    entry = greek.lookup("λόγος")          # the full structured LSJ entry
    print("\nλόγος —", len(entry.senses), "senses; the first:", entry.senses[0].text[:42])
    entry   # renders as a tidy definition card in Jupyter

### Three more levers · inflection, register, rarity

The same gold data drives three more opt-in tools. **Inflection synthesis** runs lemmatization *backwards*: `greek.inflect` maps a lemma plus a feature spec to the attested form(s) in the AGDT, and `greek.paradigm` lists the whole paradigm. **`greek.usage`** reads a word's dialect and register markers off its LSJ entry. And **`greek.terminology_rarity`** scores how unusual each word is *relative to a reference corpus* (here the Greek NT), a translation-difficulty signal that surfaces rare or technical vocabulary.

> ⬇️ **HEAVY (OPTIONAL).** Reuses the treebank, LSJ, and NT backends. Runs only if `RUN_HEAVY = True`. `usage` and `rarity` are **EXPLORATORY** aids (heuristic markers, corpus-relative scores), not measured truth.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True for inflection synthesis, usage tags, and rarity.")
else:
    greek.use_treebank()        # the inverse index + rarity's lemmatizer both lean on it

    # 1. Inflection synthesis — the lemmatizer run BACKWARDS (attested AGDT forms):
    greek.use_inflector()                                   # one-time inverse-index build, then cached
    print("λόγος, gen sg →", greek.inflect("λόγος", case="gen", number="sg"))   # ('λόγου',)
    print("paradigm cells:", len(greek.paradigm("λόγος")), "— e.g. acc sg",
          greek.inflect("λόγος", case="acc", number="sg"))                       # 14 — e.g. ('λόγον',)

    # 2. Dialect / register, read off the LSJ entry (heuristic, EXPLORATORY):
    greek.use_lsj()
    u = greek.usage("ἄναξ")
    print("ἄναξ →", "dialects:", u.dialects or "—", "| registers:", u.registers)
    # ἄναξ → dialects: — | registers: ('tragic', 'comic', 'poetic')  — a poetic word for 'lord'

    # 3. Terminology rarity, RELATIVE to the Greek NT — the rare word stands out:
    nt = greek.load_nt()
    r = greek.terminology_rarity("τὸν ἄρτον ἡμῶν τὸν ἐπιούσιον δὸς ἡμῖν", nt)   # Matt 6:11
    print("rarity:", round(r.overall, 3), "| hardest:",
          [(w.word, w.label) for w in r.hardest(2)])
    # rarity: 0.318 | hardest: [('ἐπιούσιον', 'rare'), ('ἄρτον', 'common')]
    # ἐπιούσιον — the famous near-hapax 'daily' (2 NT hits) — surfaces as the hardest word.

### Rung 5 (the last) · Seeing the syntax

> ⏱️ **HEAVY (OPTIONAL) — Dependency parser.** First run fetches the **prebuilt model** from the shared AGDT bundle (trains ~2-3 min locally, pure Python, only if the asset is unreachable). Runs only if `RUN_HEAVY = True`.

This is an **honest baseline**, not a research-grade parser. Ancient Greek is richly *non-projective* (only ~31% of AGDT sentences are projective) and arc-eager builds only projective trees. It gives clean trees for main-clause syntax, with the gold **AGDT/Prague** labels (SBJ, OBJ, PRED, ATR, ADV, Aux…). We'll parse the opening of John (a clean main clause), and `greek.evaluate_parser()` reports the real accuracy so you're never misled.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the prebuilt parser model.")
else:
    greek.use_treebank()        # optional — better POS/lemmas feed the parser
    greek.use_parser()          # prebuilt model on first use (local train is the fallback)

    tree = greek.parse("ἐν ἀρχῇ ἦν ὁ λόγος")   # John 1:1a
    print(tree)
    # 1 ἐν     ADP  AuxP ->3(ἦν)
    # 2 ἀρχῇ   NOUN ADV  ->1(ἐν)
    # 3 ἦν     VERB PRED ->0(ROOT)
    # 4 ὁ      DET  ATR  ->5(λόγος)
    # 5 λόγος  NOUN SBJ  ->3(ἦν)
    print("root:", tree.root().form,
          "| dependents of ἦν:", [t.form for t in tree.children(3)])  # ἦν | ['ἐν', 'λόγος']

    # The honest scorecard on held-out AGDT:
    print(greek.evaluate_parser())
    # ~0.51 UAS / 0.42 LAS across all text (~0.67 / 0.57 on the projective subset)

**Where we are.** From a bare line of epic Greek we read it, scanned the metre, met an honest baseline — and then, with three one-line opt-ins, got attested lemmas, dictionary senses, and a syntax tree. Each backend has a matching `greek.disable_treebank()` / `disable_lsj()` / `disable_parser()` to drop back to the pure-offline core. Network is needed only on the first call of each.

## Act 3 · The other side: Linear A, where the tools give *leads* (offline, EXPLORATORY)

Same researcher, opposite problem. **Linear A is undeciphered** — we can transliterate the signs but we don't know the language. So pyaegean's Aegean analysis never claims answers: every method here is labelled **EXPLORATORY** and surfaces patterns worth a human's attention.

We'll do one concrete thing — check whether a 3,500-year-old account *adds up* — then let the corpus suggest its own structure. The bundled corpus is the Linear A Workbench dataset (transliterated, with numerals and find-spot metadata) and loads **offline**.

In [ ]:
corpus = aegean.load("lineara")        # 1,721 inscriptions, bundled, offline
print(len(corpus), "inscriptions")

doc = corpus.get("HT13")               # a well-known account from Haghia Triada
print("words   :", [t.text for t in doc.words])
print("numerals:", [t.text for t in doc.numerals])
print("site    :", doc.meta.site, "| period:", doc.meta.period)   # Haghia Triada | LMIB
doc   # in Jupyter the Document renders as a glyph/transcription card

### Does the arithmetic close?

Many Linear A tablets are accounts: line items followed by a total word, **KU-RO**. `balance_check` sums the items a total governs and compares them to the stated total.

> **EXPLORATORY.** Section boundaries are heuristic and Linear A metrology is genuinely contested. This *finds* lines worth a human's attention — it is not a verdict.

In [ ]:
from aegean.analysis import balance_check

for chk in balance_check(doc):
    print(chk)
# BalanceCheck(stated_total=130.5, computed_sum=131.0, item_count=6,
#              difference=0.5, balances=False, marker='KU-RO', total_line_index=7)
#
# The items sum to 131.0 but the scribe wrote 130.5 — a ½-unit gap. Ancient error?
# misread sign? an artefact of where *we* drew the section? A lead for a human.

### How does the total-word behave across the corpus?

Two composable tools. The **sign-pattern** language searches by shape (`*` = *exactly one sign*), so `KU-*-RO` matches a three-sign word — and `KU-RO` itself does **not** match (no middle sign). The **query engine** then combines surface conditions with `and`/`or`/`not`. (Unlike the interpretive tools, these predicates are deterministic filters, not exploratory.)

In [ ]:
from aegean.analysis import word_matches_sign_pattern, FilterRow, run_query

# Words shaped KU-?-RO (one sign between):
print([(w, c) for w, c in corpus.word_frequencies()
       if word_matches_sign_pattern(w, "KU-*-RO")])
# [('KU-MA-RO', 1)]

# Haghia Triada tablets (id contains 'HT') that contain the exact word KU-RO:
res = run_query(corpus, [
    FilterRow("id-contains", "HT"),
    FilterRow("ins-contains-word", "KU-RO", connector="and"),
], output="inscriptions")
print(len(res.inscriptions), "tablets:", [d.id for d in res.inscriptions][:8])
# 32 tablets: ['HT9a','HT9b','HT11a','HT11b','HT13','HT25b','HT27a','HT39']

### A minimal pair? KU-RO vs KI-RO

The second-most-common accounting word is **KI-RO** (often read as a *deficit/owed* counterpart to the KU-RO total). They differ in one vowel — pyaegean can make that intuition quantitative with a **weighted phonetic edit distance** (0 = identical → 1) and a per-phoneme **alignment**.

> **EXPLORATORY.** The phonetic values come from the conventional sign→sound transliteration of an *undeciphered* script. A small distance is a *lead*, never a sound law — which is exactly why the scheme is configurable.

In [ ]:
from aegean.scripts.lineara.phonetic import word_to_phonetic
from aegean.analysis import phonetic_distance, align_phonetic

a, b = word_to_phonetic("KU-RO"), word_to_phonetic("KI-RO")
print(a, b)                                  # kuro kiro
print("distance:", round(phonetic_distance(a, b), 3))   # 0.075  (one cheap vowel swap)

for cell in align_phonetic(a, b):
    print(f"  {cell.a or '—'}  {cell.b or '—'}  {cell.op}")
#   k  k  match
#   u  i  sub-vowel    ← the single, low-cost difference
#   r  r  match
#   o  o  match

### Letting the corpus suggest morphology

With no known grammar, we can still ask a purely distributional question: which words share a stem and differ by a **productive ending** (a final sign-string that recurs across many distinct words)? `find_morphological_clusters` answers it. The top cluster is the famous libation-formula family **JA-SA-SA-RA-ME**.

> **EXPLORATORY.** A "suffix" here is a recurring final sign-string, **not** a confirmed morpheme. These are candidate paradigms for a morphologist to weigh.

In [ ]:
from aegean.analysis import find_morphological_clusters

clusters = find_morphological_clusters(corpus.word_frequencies())
print(len(clusters), "candidate clusters")     # 81

fam = clusters[0]                               # the JA-SA family
print("stem:", fam.stem, "| total occurrences:", fam.total_count)
for m in fam.members:
    print(f"   {m.word:<16} x{m.count:<3} +{m.suffix or '(stem)'}")
# stem: JA-SA | total occurrences: 16
#    JA-SA-SA-RA-ME   x7   +SA-RA-ME
#    JA-SA            x4   +(stem)   ... etc.

### Do KU-RO and KI-RO travel together?

If KI-RO is the deficit counterpart to the KU-RO total, they should **co-occur** on the same tablets more than chance. pyaegean ships the corpus-linguistics association tests — log-likelihood ratio (Dunning G²) and Fisher's exact — over a 2×2 contingency table.

> **EXPLORATORY.** A strong association is a distributional fact worth explaining — it does not by itself confirm the semantic reading.

In [ ]:
from aegean.analysis import log_likelihood_ratio_2x2, fishers_exact

N = len(corpus)
presence = [{t.text for t in d.words} for d in corpus]
count = lambda w: sum(w in s for s in presence)

ca, cb = count("KU-RO"), count("KI-RO")
joint = sum(("KU-RO" in s and "KI-RO" in s) for s in presence)
print(f"docs: KU-RO={ca}  KI-RO={cb}  both={joint}  (of {N})")   # 34  12  5  (of 1721)
print("Dunning G² :", round(log_likelihood_ratio_2x2(joint, ca, cb, N), 2))  # 23.94
print("Fisher  p  :", f"{fishers_exact(joint, ca, cb, N):.2e}")               # 1.60e-06
# A strong, very-unlikely-by-chance association — a real lead for the deficit reading.

### Is the structure beyond chance? · explicit null models (EXPLORATORY)

The clusters and collocations above are real distributional facts, but some apparent structure is only an artefact of how often each sign happens to occur. The structure tooling makes that check explicit: **`candidate_morphs`** ranks recurring word-final sign-strings (candidate suffixes), and **`monte_carlo_p`** scores a structure statistic against a *stated* null. Here that null pools the signs and re-deals them into words of the same lengths, which keeps each sign's overall frequency but destroys ordering. (Companion tools `sign_embeddings` and `induce_classes` surface distributional sign neighbourhoods and classes the same way.)

> **EXPLORATORY.** A small p-value says the statistic departs from *that* null, never that a reading is correct. These rank leads for a human; they do not decipher.

In [ ]:
from aegean.analysis import candidate_morphs, monte_carlo_p
from collections import Counter

words = [t.text for d in corpus for t in d.words if "-" in t.text]   # 1,381 multi-sign words

# Recurring word-final sign-strings — candidate suffixes (EXPLORATORY):
print("top final morphs:", candidate_morphs(words, min_count=8)[:5])
# [('TE', 37), ('NA', 36), ('JA', 34), ('RE', 34), ('TI', 29)]

# Is the commonest ending more frequent than sign frequency alone would yield?
top_ending = lambda ws: max(Counter(w.split("-")[-1] for w in ws).values())
res = monte_carlo_p(top_ending(words), top_ending, words, null="within_word", n=999, seed=0)
print(res)
# MonteCarloResult(observed=78, p_value=0.005, null_mean=62.5, ..., null='within_word')
# 78 words share the commonest ending vs ~62 under reshuffling (p ≈ 0.005): the word-final
# regularity is beyond chance, a real lead for a morphologist, not a frequency artefact.

## Act 4 · Scale: the full Linear B corpus, one call away

The acts above worked on one line and on the bundled corpora. The same toolkit loads the **complete Mycenaean corpus** — DAMOS, the Database of Mycenaean at Oslo (~5,900 tablets, CC BY-NC-SA 4.0, fetched ~2 MB to your cache) — and the corpus statistics answer real questions about it. The classic: *what vocabulary makes Pylos different from the rest of the Mycenaean world?*

> ⬇️ **HEAVY (OPTIONAL)** — fetches ~2 MB on first run. Runs only if `RUN_HEAVY = True`.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the DAMOS corpus (~2 MB).")
else:
    from aegean.analysis import keyness

    damos = aegean.load("damos")                  # ~5,900 tablets, then cached
    pylos = damos.filter(site="Pylos")
    rest = [d for d in damos.documents if d.meta.site != "Pylos"]

    for row in keyness(pylos, rest)[:5]:          # G² + log-ratio, strongest first
        print(f"{row.item:10}  G²={row.log_likelihood:6.1f}  "
              f"log-ratio={row.log_ratio:+.1f}  ({row.target_count}× at Pylos)")
    # pe-mo ('seed'), o-na-to ('lease plot'), ko-to-na ('land plot') … — the Pylos
    # land-tenure series surfaces immediately: the textbook Ventris & Chadwick result.

    print()
    print("cite it:", pylos.cite()[:90], "…")

# Also there when you want them: aegean.load("sigla") (the SigLA Linear A dataset),
# analysis.dispersions() (Gries' DP), and `aegean plot keyness damos --site Pylos -o out.png`.

### Interchange · EpiDoc in and out (offline)

The corpus model speaks **EpiDoc TEI**, the standard for digital epigraphy, in *both* directions with no extra dependency: `io.write_epidoc` exports, and `io.from_epidoc` reads any EpiDoc edition (a file or a folder of `.xml`) back into a corpus. So your own inscriptions, or any of the 400+ published EpiDoc corpora, round-trip through pyaegean entirely offline.

In [ ]:
import tempfile, os
from aegean import io as aio

# write the bundled Linear A 'Zakros' tablets to EpiDoc, then read them straight back;
# only the stdlib XML parser is used, no [epidoc] extra:
epi = os.path.join(tempfile.mkdtemp(), "epidoc")
aio.write_epidoc(aegean.load("lineara").filter(site="Zakros"), epi)
back = aio.from_epidoc(epi, script_id="lineara")

doc = back.documents[0]
print(f"read {len(back)} documents back from EpiDoc")
print(f"first: {doc.id} · {doc.meta.site} · {len(doc.tokens)} tokens")
print("tokens:", " ".join(t.text for t in doc.tokens[:6]))
# read 53 documents back from EpiDoc
# first: ZA10a · Zakros · 16 tokens
# tokens: TA-NA-TE 2 PA 1 A-KU-MI-NA 1
# From the shell: aegean import <dir> --epidoc --script lineara -o corpus.json

## Coda · Grounded, EXPLORATORY translation

pyaegean's AI layer (`aegean.ai` / `aegean.translate`) is **multi-provider** (Anthropic / OpenAI / Grok / Gemini / OpenRouter) and only generates after a **local, deterministic grounding** step builds evidence from the very tools above. The grounding needs no key or network, so we can inspect it here; the generative call needs a provider extra (e.g. `pyaegean[anthropic]`) and an API key, and is wrapped so a missing SDK or key is handled gracefully rather than crashing.

When the LSJ backend is switched on, that grounding also carries **gated dictionary glosses** (`translate(..., glosses=True)`, on by default): a polysemy gate leaves highly ambiguous words to the model, where forcing a single dictionary sense would mislead, and feeds a gloss only where a dominant sense genuinely helps. Pass `glosses=False` for lemma-only grounding.

> Any AI reading is a **hypothesis carrying its provenance**, explicitly labelled EXPLORATORY — never asserted truth. For undeciphered Linear A especially, it is a lead for a human expert.

In [ ]:
from aegean import ai, translate

print("providers:", ai.list_providers())   # ['anthropic', 'gemini', 'grok', 'local', 'openai', 'openrouter']

# The deterministic, LOCAL grounding evidence — built from the pipeline above, no key
# or network. The default mode is "morphology": lemma, part of speech, voice, case-role,
# and rare-word flags, the analysis a model most often gets wrong:
for item in translate.grounding_for("μῆνιν ἄειδε θεά", "greek"):
    print("  ", item)
for item in translate.grounding_for("KU-RO KI-RO", "lineara"):
    print("  ", item)   # KU-RO → /kuro/   ·   KI-RO → /kiro/

# The generative step is optional — it needs a provider SDK + an API key. Handled gracefully:
try:
    result = translate.translate("μῆνιν ἄειδε θεά", script="greek")  # grounded; reads its key from the env
    print(result.labeled())          # carries the [EXPLORATORY · translate · …] tag + provenance
except (ai.ProviderNotInstalled, ai.MissingAPIKey) as e:
    print(f"Generative step skipped ({type(e).__name__}). The grounding above is fully local.")

### Choosing the grounding for the job

`translate(..., mode=...)` controls how much analysis the model is handed, and the right amount depends on the passage:

| mode | the model is told | reach for it when |
|---|---|---|
| `"morphology"` *(default)* | lemma, POS, **voice**, case-role, clause skeleton, rare-word flags | always: the safe floor, it fixes wrong voice, swapped subject/object, and case errors |
| `"full"` | morphology **plus concise glosses** on the rare words | rare, technical, or poetic vocabulary the model will not know |
| `"none"` | nothing | a strong model on a famous, easy passage |

The deterministic morphology helps in every regime; the `"full"` glosses help on rare vocabulary, and both help **weaker models and rarer text the most**. Grounding does not make a weak model an expert, and Ancient Greek translation has real interpretive range (no two published Homers agree), so the result is always a labeled, provenanced hypothesis.

In [ ]:
text = "καὶ ἡγοῦμαι σκύβαλα εἶναι, ἵνα Χριστὸν κερδήσω."   # Philippians 3:8 — σκύβαλα is rare

# mode="morphology" (the default) needs no download — lemma, POS, voice, case-role, rare flags:
print("morphology grounding:")
for item in translate.grounding_for(text, "greek", mode="morphology"):
    print("  ", item)

# mode="full" adds a concise gloss on the RARE words — the meaning a small model would miss
# (σκύβαλα → 'refuse', not 'scourges'). Glosses come from concise, common-sense-first
# dictionaries, gated to rare words — never LSJ's archaic first sense. Needs a dictionary fetch:
if not RUN_HEAVY:
    print("\n[skipped] set RUN_HEAVY = True to fetch a dictionary and gloss the rare σκύβαλα.")
else:
    from aegean import greek
    greek.use_lsj(); greek.use_lexicon("abbott-smith")   # a concise NT dictionary
    print("\nfull grounding (rare words glossed):")
    for item in translate.grounding_for(text, "greek", mode="full"):
        print("  ", item)
    # … σκύβαλα (σκύβαλον): refuse   ← the lexical help, exactly where it is needed

## What you can do now

In one sitting you took **one line of Homer** down the whole ladder — syllables, accent, IPA, metre, POS, morphology — then, with a few opt-in calls, lifted POS/lemma to 100% on attested forms, *generated* lemmas for unseen ones with the neural model, glossed from the LSJ, and read a dependency tree. Then you turned the same toolkit on an **undeciphered** script to check a Bronze-Age ledger, quantify a minimal pair, mine a word-family, and measure a collocation — generating *leads*, not answers.

Every stage is an independent function, so you can lift just the rung you need onto **your own** text — or skip the ladder entirely: `greek.pipeline(text)` runs the whole stack in one call, `greek.use_neural_pipeline()` swaps in the most accurate model, `greek.load_work("tlg0012.tlg001")` fetches the complete Iliad as a corpus, `aegean.load("damos")` the full Linear B corpus, and `pip install "pyaegean[cli]"` gives you all of it from the command line (`aegean --help`).

**Where next**
- [Recipes](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes) — end-to-end scholarly workflows: reconcile a whole corpus's accounting, map a word, lemmatize-and-cite a chapter, keyness, cross-script triage.
- [Greek NLP](https://github.com/ryanpavlicek/pyaegean/wiki/Greek-NLP) — the full reference (prosody, the benchmark harness, held-out and out-of-AGDT evaluation, and the opt-in backends in detail).
- [Analysis](https://github.com/ryanpavlicek/pyaegean/wiki/Analysis) — phonetic distance/alignment, morphological clustering, collocation statistics, the query engine.
- [Data & Provenance](https://github.com/ryanpavlicek/pyaegean/wiki/Data-and-Provenance) — what's bundled vs fetched, and the CC BY-SA attribution for the treebank and LSJ.

> The throughline: the toolkit tells you where it's confident and where it's guessing. For deciphered Greek that's an honest baseline you can upgrade to attested scholarship; for the undeciphered Linear A material it's always **EXPLORATORY** — leads for a human expert, never ground truth.

## Appendix · The whole toolkit at a glance

The narrative above walked one Greek line and one undeciphered ledger. This appendix is the compact reference: each section shows one more part of the toolkit in a single short cell, and every cell runs without network access. The two exceptions follow the notebook's conventions: a `RUN_HEAVY`-gated fetch, and code that is shown but not run. The cells run top to bottom; the first section loads the bundled Linear A corpus as `la` for the sections that reuse it.

### The compound query engine

`Corpus.query` (and its functional twin `analysis.run_query`, used in Act 3) chains typed where-predicates over documents and words: 16 fields from `id-contains` to `word-sign-pattern`, AND/OR connectors, and results that carry a human-readable description of the search plus their own citation.

In [ ]:
from aegean.analysis import FilterRow

la = aegean.load("lineara")     # the appendix reuses this corpus as `la`

res = la.query([
    FilterRow("site-is", "Haghia Triada"),
    FilterRow("word-suffix", "TE", connector="and"),
    FilterRow("word-cooccurs-with", "KU-RO", connector="and"),
], output="words")
print(res.words)          # [('MA-KA-RI-TE', 2), ('DE-ME-TE', 1)]
print(res.description)    # Site is: Haghia Triada · Word ends with: TE · Word co-occurs with: KU-RO
print(res.cite()[:74], "…")     # query results cite exactly what was searched

### Dispersion and keyness

Two corpus-linguistic staples, offline (collocation statistics appeared in Act 3). `dispersions` computes Gries' DP: is a word spread evenly through the corpus or clumped in a few documents? `keyness` (G² with a log-ratio effect size) asks what is characteristic of one subcorpus against another.

In [ ]:
from aegean.analysis import dispersions, keyness

# DP near 1 = concentrated in few documents; near 0 = spread evenly:
for d in dispersions(la, min_frequency=10)[:3]:
    print(f"{d.item:8} freq={d.frequency:3}  range={d.range:3}  DP={d.dp_norm:.2f}")
# KU-RO    freq= 37  range= 34  DP=0.85
# KI-RO    freq= 16  range= 12  DP=0.94
# SA-RA₂   freq= 20  range= 20  DP=0.95

# What is characteristic of Haghia Triada against the rest of the corpus?
ht = la.filter(site="Haghia Triada")
rest = [doc for doc in la.documents if doc.meta.site != "Haghia Triada"]
for row in keyness(ht, rest)[:3]:
    print(f"{row.item:8} G²={row.log_likelihood:5.1f}  log-ratio={row.log_ratio:+.1f}  ({row.target_count}× at HT)")
# KU-RO    G²= 35.2  log-ratio=+4.1  (35× at HT)  — the accounting vocabulary IS the HT signature
# SA-RA₂   G²= 27.2  log-ratio=+5.3  (20× at HT)
# KI-RO    G²= 21.7  log-ratio=+5.0  (16× at HT)

### Document types and scribal hands

`classify_structure` labels one document's internal shape (accounting, list, label, and so on), `document_type_profile` profiles the physical supports the script travels on, and `scribal_hands` aggregates the corpus's attested scribe attributions.

In [ ]:
from aegean.analysis import classify_structure, document_type_profile, scribal_hands

print("HT13 is:", classify_structure(la.get("HT13")))    # accounting

for p in document_type_profile(la)[:3]:
    print(f"  {p.type:8} ×{p.count:<4} {p.share_pct:4.1f}%  numerals on {p.numerals_pct:4.1f}%")
#   Nodule   ×886  51.5%  numerals on  0.8%   — sealings: names, rarely numbers
#   Tablet   ×393  22.8%  numerals on 77.9%   — the ledgers live on tablets
#   Roundel  ×151   8.8%  numerals on  4.0%

for h in scribal_hands(la)[:2]:
    print(f"  {h.hand}: {h.doc_count} documents at {list(h.sites)}")
#   HT Wa Scribe 10: 106 documents at ['Haghia Triada']
#   HT Wa Scribe 84: 41 documents at ['Haghia Triada']

### Aegean numerals and ledger anatomy

The numeral layer parses token values (integers and the Aegean fraction signs) and formats them back; `parse_account_lines` reads a tablet as a ledger, assigning each line a role. Act 3's `balance_check` is built on exactly this anatomy.

In [ ]:
from aegean.core import numerals

print(numerals.parse_value("130"), numerals.parse_value("¹⁄₂"), numerals.format_value(130.5))
# 130 0.5 130½

doc = la.get("HT13")
lines = [[doc.tokens[i].text for i in line] for line in doc.lines]
for al in numerals.parse_account_lines(lines):
    print(f"  line {al.index}: {al.role:6} {' '.join(al.tokens):24} = {al.value}")
#   line 0: header KA-U-DE-TA VIN 𐄁 TE 𐄁    = 0.0
#   line 1: item   RE-ZA 5 ¹⁄₂              = 5.5
#   …
#   line 7: total  KU-RO 130 ¹⁄₂            = 130.5   ← the KU-RO total Act 3 audited

### Sign inventories and the deciphered bridges

Every script plugin exposes its full sign inventory (glyph, transliteration label, sound value where read, catalogue metadata). For the two deciphered syllabaries, Linear B and Cypriot, a lexicon bridge maps a syllabic spelling to its Greek word and gloss; Act 3 used the Linear A counterpart, where the sound values are exploratory.

In [ ]:
from aegean import get_script
from aegean.scripts import linearb, cypriot

for sid in aegean.registered_scripts():
    print(f"  {sid:12} {len(get_script(sid).sign_inventory.signs):3} signs")
#   cypriot       55 · cyprominoan  99 · greek  25 · lineara 342 · linearb 211

ka = get_script("linearb").sign_inventory.by_label("KA")
print(ka.glyph, ka.label, "·", ka.phonetic, "· Bennett", ka.attrs["bennett"])   # 𐀏 KA · ka · Bennett B077

# Deciphered scripts write Greek — bridge a spelling to its Greek word:
print(linearb.word_to_phonetic("ti-ri-po"), linearb.greek_reading("ti-ri-po"))
# tiripo ('τρίπους', 'tripod')
print(cypriot.word_to_phonetic("pa-si-le-u-se"), cypriot.greek_reading("pa-si-le-u-se"))
# pasileuse ('βασιλεύς', 'king')

### Cypriot: IG XV 1 with its editorial apparatus

The bundled Cypriot corpus carries the IG XV 1 syllabic inscriptions, and the edition's Leiden apparatus survives the trip into Python: every token bears a `ReadingStatus` (CERTAIN, UNCLEAR, RESTORED, LOST), so an analysis can weigh a restored royal title differently from one read on the stone.

In [ ]:
from aegean.scripts import cypriot

cyp = aegean.load("cypriot")            # 180 documents, bundled
d = cyp.get("IG XV 1, 120")
print(d.id, "·", d.meta.site, "·", d.meta.period)   # IG XV 1, 120 · Kurion · um 710-675 (I) / nach 498 (II)
for t in d.tokens:
    print(f"  {t.text:18} {t.status.name}")
#   a-ke-se-to-ro      CERTAIN      ← read on the stone
#   to                 CERTAIN
#   pa-po              CERTAIN
#   pa-si-le-wo-se     RESTORED     ← 'of the king': supplied by the editor
#   ti-mo-ke-re-to-se  RESTORED
#   e-mi               RESTORED
print("e-mi →", cypriot.greek_reading("e-mi"))      # ('εἰμί', 'I am')

### The Greek New Testament: gold annotations

Every NT token carries a gold lemma, Robinson morphology, Strong's number, and a reconciled UD part of speech; `use_dodson` adds the bundled Dodson Koine lexicon for glossing. The full 27-book corpus fetches once (~16 MB) via `greek.load_nt()` or `aegean.load("nt")`, and in this notebook the Coda's grounding step has already cached it. With no network access at all, `load_nt` falls back to the bundled offline sample (John 1 + Philemon), which carries the same annotations, so this cell runs either way.

In [ ]:
john = greek.load_nt("John", ref="1.1")     # book + chapter.verse addressing
for t in john.documents[0].tokens[:5]:
    a = t.annotations
    print(f"  {t.text:8} {a['lemma']:6} {a['morph']:9} Strong's {a['strongs']}")
#   Ἐν       ἐν     PREP      Strong's 1722
#   ἀρχῇ     ἀρχή   N-DSF     Strong's 746
#   ἦν       εἰμί   V-IAI-3S  Strong's 1510
#   ὁ        ὁ      T-NSM     Strong's 3588
#   Λόγος,   λόγος  N-NSM     Strong's 3056

greek.use_dodson()                          # bundled Koine lexicon, offline
print("gloss_nt('λόγος') →", greek.gloss_nt("λόγος"))          # a word, speech, divine utterance, analogy
print("lookup_nt('χάρις') →", greek.lookup_nt("χάρις").gloss)  # grace, favor, kindness

### Idiom glosses

A bundled lexicon of 38 non-compositional idioms grounds phrases whose sense no word-by-word gloss captures (ἐφ' ἡμῖν is not "upon us"). The AI translation layer injects these automatically; they are equally usable on their own.

In [ ]:
from aegean import ai

for g in ai.idiom_glosses("τοῦτο οὐκ ἔστιν ἐφ᾽ ἡμῖν"):
    print(g.content, "·", g.source)
# ἐφ' ἡμῖν: in our power, up to us · lexicon:idiom

### Lenient normalization and syllable weights

`normalize(lenient=True)` repairs the classic OCR confusables (Latin letters inside Greek words) and issues a warning naming exactly what it changed. `syllable_quantities` returns each syllable's metrical weight, the raw material of Act 1's scanner. Accent placement and sandhi resolution appeared in Act 1.

In [ ]:
print(greek.normalize("λoγoς", lenient=True))    # λογος — both Latin 'o's repaired (a warning says so)
print(greek.syllable_quantities("ἄνθρωπος"))     # ['heavy', 'heavy', 'heavy']
print(greek.syllable_quantities("λόγος"))        # ['light', 'heavy']

### The other metres

Act 1 scanned epic hexameter. The same scanner reads the elegiac pentameter, the iambic trimeter of drama (with caesura detection and resolution), and the aeolic lyric templates.

In [ ]:
sc = greek.scan_pentameter("κείμεθα τοῖς κείνων ῥήμασι πειθόμενοι.")   # Simonides, for the Spartan dead
print(sc.pattern)                       # —⏑⏑|——|—|—⏑⏑|—⏑⏑|×

sc = greek.scan_trimeter("ὦ κοινὸν αὐτάδελφον Ἰσμήνης κάρα")           # Sophocles, Antigone 1
print(sc.pattern, "· caesura:", sc.caesura)   # ×—⏑—|×—⏑—|×—⏑× · caesura: hephthemimeral

sc = greek.scan_aeolic("φαίνεταί μοι κῆνος ἴσος θέοισιν",              # Sappho 31.1
                       "sapphic_hendecasyllable")
print(sc.pattern)                       # —⏑—×—⏑⏑—⏑—×

print(sorted(greek.AEOLIC_LINES))       # the supported aeolic templates
# ['adonean', 'alcaic_decasyllable', 'alcaic_enneasyllable', 'alcaic_hendecasyllable',
#  'glyconic', 'pherecratean', 'sapphic_hendecasyllable']

### The works catalogue, offline

The catalogue of ~1,800 Perseus and First1KGreek works is bundled metadata: search it offline by author, title, or free text, then fetch any entry with `greek.load_work(id)` (network).

In [ ]:
print(len(greek.catalog()), "works in the offline catalogue")   # 1778
for w in greek.catalog(title="Symposium"):
    print(f"  {w['id']}  {w['author']}: {w['title']}")
#   tlg0032.tlg004  Xenophon: Symposium
#   tlg0059.tlg011  Plato: Symposium
#   tlg0062.tlg015  Lucian of Samosata: Symposium
#   tlg2959.tlg001  Methodius: Symposium Sive Convivium Decem Virginum
# then: greek.load_work("tlg0059.tlg011")   (fetches and caches the full text)

### Geography

A bundled, Pleiades-aligned gazetteer covers the Aegean find-sites, with contested attributions flagged as such. The plain dictionary needs nothing beyond the core; with the `[geo]` extra installed, `geo.word_distribution(corpus, word)` and `geo.to_geodataframe(...)` return GeoDataFrames ready for mapping.

In [ ]:
from aegean import geo

sites = geo.site_coordinates()
print(len(sites), "find-sites")                                  # 56 find-sites
sc = sites["Haghia Triada"]
print(sc.name, "·", sc.lat, sc.lon, "·", sc.region, "· Pleiades", sc.pleiades)
# Haghia Triada · 35.06 24.79 · crete · Pleiades 589672

### Citation, provenance, and fingerprints

Every corpus carries provenance and cites itself in plain, APA, or BibTeX form; a subset says that it is one. `fingerprint()` is a stable content hash over the actual data, so two analyses can prove they saw the same corpus.

In [ ]:
print(la.cite())
# Godart, L. & Olivier, J.-P. (1976–1985). Recueil des inscriptions en linéaire A. — https://github.com/mwenge/lineara.xyz
print(la.provenance.apa().split(" [")[0])       # the APA form (trailing note trimmed here)
print(la.provenance.bibtex().splitlines()[0])   # @misc{aegean-corpus,

sub = la.subset(["HT13", "HT9a"])
print(sub.cite()[-35:])                         # [subset: 2 of 1721 documents by id]
print(la.fingerprint()[:16], "≠", sub.fingerprint()[:16])
# c6e3d680a3a85842 ≠ a41bff976f45c5ea

### Evaluation receipts

`eval_receipt` seals a measured score together with the package version and a manifest of every data artifact in play under a content-addressed id: a reproducibility receipt for any number you publish. Here, the offline lemmatizer is scored against Philemon's gold lemmas from the previous section.

In [ ]:
phm = greek.load_nt("Philemon").documents[0]
gold = [(t.text, t.annotations["lemma"]) for t in phm.tokens if t.annotations.get("lemma")]
acc = sum(1 for w, lem in gold if greek.lemmatize(w) == lem) / len(gold)

receipt = greek.eval_receipt(
    {"lemma_acc": round(acc, 4)},
    treebank="nt-sample", split="Philemon",
    protocol="offline greek.lemmatize vs Nestle 1904 gold lemmas",
)
print(receipt.scores, "· id", receipt.id)
# {'lemma_acc': 0.6328} · id …   (the id hashes scores + package version + data manifest)

### Save, search, stream: exports and the database

The corpus model round-trips losslessly through JSON and EpiDoc (Act 4 showed the EpiDoc round trip), exports to CSV (and Parquet with the `[parquet]` extra), and persists to SQLite with full-text search: `db.search` for instant lookups, `db.stream` to iterate a large corpus one document at a time.

In [ ]:
import os, tempfile
from aegean import db, io

zak = aegean.load("lineara").filter(site="Zakros")
print("to_json  :", len(zak.to_json()), "chars")                 # 248192 chars (a string; pass a path to write)
print("to_epidoc:", io.to_epidoc(zak.documents[0]).splitlines()[0])   # <?xml version='1.0' encoding='UTF-8'?>

tmp = tempfile.mkdtemp()
path = os.path.join(tmp, "zakros.db")
db.to_sqlite(zak, path)                                          # SQLite + FTS index
print("db.search:", db.search(path, "KU-RO"))                    # [('ZA15b', 5, 'KU-RO')]
print("db.stream:", sum(1 for _ in db.stream(path)), "documents, one at a time")
io.to_csv(zak, os.path.join(tmp, "zakros.csv"))                  # one row per document

### Combine and import your own texts

`Corpus.from_records` builds a corpus from plain dictionaries, `io.from_csv` imports a spreadsheet (`io.from_text`, `from_text_dir`, and `from_epidoc` cover the other arrival routes), and `combine` merges corpora with provenance that names every ingredient.

In [ ]:
from aegean import Corpus, combine

mine = Corpus.from_records([{"id": "graffito-1", "text": "γνῶθι σεαυτόν"}], script_id="greek")

csv_path = os.path.join(tmp, "finds.csv")
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("id,text,site\nfind-1,ἐν ἀρχῇ ἦν ὁ λόγος,Ephesus\n")
imported = io.from_csv(csv_path, id_col="id", meta_cols=("site",))

both = combine([mine, imported])                # dedupe="error" guards against id collisions
print(len(both), "documents:", [d.id for d in both.documents])
# 2 documents: ['graffito-1', 'find-1']
print(both.provenance.citation)
# Merged corpus of: User-supplied corpus (Corpus.from_records); Imported CSV: finds.csv

### The epigraphic shelf

Beyond the bundled corpora and Act 4's DAMOS, five Greek inscription corpora fetch on demand, each a sha256-pinned asset with license and provenance attached:

```python
isi   = aegean.load("isicily")   # 2,855 Greek inscriptions of Sicily (CC BY 4.0)
iip   = aegean.load("iip")       # 2,113 of Israel/Palestine (CC BY-NC 4.0)
iospe = aegean.load("iospe")     # 1,194 of the northern Black Sea coast (CC BY 4.0)
igcyr = aegean.load("igcyr")     # 997 of Cyrenaica: archaic Doric, verse (CC BY-NC-SA 4.0)
edh   = aegean.load("edh")       # 1,286 Imperial-era Greek of the EDH (CC BY-SA 4.0)
```

The registry itself is inspectable offline, and the next cell fetches one small corpus under the notebook's `RUN_HEAVY` gate.
Every token of these corpora carries the editor's **reading status** (`CERTAIN` / `UNCLEAR` / `RESTORED` / `LOST`) and an `edition_fidelity` flag in the provenance, so restored or damaged readings are never mistaken for securely-read text. The DDbDP documentary papyri (the next section) carry the same. See the wiki's *Using critical editions* for the full model.


In [ ]:
from aegean import data

names = sorted(data.versions()["fetched"])
print(len(names), "fetchable datasets, all sha256-pinned:")
for i in range(0, len(names), 5):
    print("  " + " · ".join(names[i:i + 5]))
# 19 fetchable datasets, all sha256-pinned:
#   abbott-smith-index · agdt-derived · cunliffe-index · damos-corpus · ddbdp-corpus
#   … (corpora, lexicon indexes, and the neural models)

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the I.Sicily corpus (~7 MB).")
else:
    isi = aegean.load("isicily")
    print(len(isi), "inscriptions ·", sum(len(d.tokens) for d in isi.documents), "tokens")
    # 2855 inscriptions · 28919 tokens
    d0 = isi.documents[0]
    print(d0.id, "·", d0.meta.site, "·", d0.meta.period)   # ISic000046 · Syracusae · 5th century CE(?)
    print(isi.cite()[:70], "…")
    # I.Sicily (ISicily/ISicily, CC BY 4.0), primary-Greek inscriptions — …
    # every token carries the editor's reading status (new in 0.29.0)
    from collections import Counter
    dist = Counter(t.status.name for d in isi.documents for t in d.tokens)
    print("reading status:", dict(dist))
    # reading status: {'CERTAIN': 22124, 'RESTORED': 5340, 'UNCLEAR': 1143, 'LOST': 312}
    print("edition fidelity:", isi.provenance.edition_fidelity)
    # edition fidelity: apparatus-preserved,normalized


### DDbDP: 57,329 documentary papyri

The Duke Databank of Documentary Papyri (papyri.info, CC BY 3.0) ships as a SQLite database with full-text search. `aegean.load("ddbdp")` materialises all ~4.4 million tokens in memory (heavy: on the order of 100 seconds); the memory-friendly paths are search and streaming, and those are what the next cell uses.

> ⬇️ **HEAVY (OPTIONAL).** The database is a one-time **~206 MB** download (~757 MB on disk), then cached. Gated on the same `RUN_HEAVY` switch as every other downloading cell.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the DDbDP database (~206 MB, one-time).")
else:
    from aegean import db
    from aegean.scripts.greek import ddbdp_db

    path = ddbdp_db()                            # fetched once, then cached
    print(db.search(path, "χαίρειν", limit=3))   # instant FTS over all 57,329 papyri
    # [('aegyptus.94.13', 1, 'χαίρειν'), ('analpap.23_24.169', 37, 'χαίρειν'),
    #  ('analpap.23_24.96', 4, 'χαίρειν')]
    for doc in db.stream(path):                  # flat-memory iteration, document by document
        print(doc.id, "·", doc.meta.name, "·", doc.meta.period)
        break
    # aegyptus.103.69_1 · AEGYPTUS 103 69_1 · AD 235

### The neural pipeline and the AI layer

> ⬇️ **HEAVY (OPTIONAL) — read before flipping the switch.** The two cells below are real, runnable code, gated on the same `RUN_HEAVY` switch as the rest of the notebook:
>
> * **Joint neural pipeline** — needs the `[neural]` extra (`pip install "pyaegean[neural]"`; on Colab run `%pip install -q "pyaegean[neural]"` in a fresh cell first) and a one-time **~173 MB** model download. Torch-free ONNX, runs on CPU.
> * **Generative translation** — calls a paid external service and needs an API key in the environment for one provider: `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `XAI_API_KEY`, `GEMINI_API_KEY`, or `OPENROUTER_API_KEY`. The output is exploratory and labeled as such. Without a key the cell prints a note instead of failing.
>
> Automated runs keep `RUN_HEAVY = False`, so both cells print `[skipped]` there.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True (with the [neural] extra installed) for the joint pipeline.")
else:
    greek.use_neural_pipeline()                  # fetches the grc-joint model once (~173 MB)
    recs = greek.pipeline("μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος", parse=True)
    print(recs[0])
    # TokenRecord(sentence=0, index=1, text='μῆνιν', upos='NOUN', lemma='μῆνις', lemma_known=True,
    #             head=2, relation='obj', xpos='n-s---fa-', feats='Case=Acc|Gender=Fem|Number=Sing')

In [ ]:
# RUN_HEAVY gates this cell; it also needs one provider API key in the environment.
_ENV = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "grok": "XAI_API_KEY",
        "gemini": "GEMINI_API_KEY", "openrouter": "OPENROUTER_API_KEY"}
_provider = next((p for p, k in _ENV.items() if os.environ.get(k)), None)
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True (and a provider key) for a generative translation.")
elif _provider is None:
    print("no provider key found — set one of", ", ".join(_ENV.values()), "and rerun.")
else:
    from aegean.ai import get_client

    result = translate.translate("μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος",
                                 verify=True, client=get_client(_provider))
    print(result.labeled())   # [EXPLORATORY · translate …] + the translation + provenance